# 实验 00：数据集清单与小样本检查

## 目标

在下载大型数据或模型前，验证候选数据集的登记信息、用途和本地状态。

成功标准：

- 登记表可以被标准库读取；
- `dataset_id` 唯一；
- 每个条目具有来源、许可、用途、划分策略和状态；
- 只预览小样本，不把大型文件整体载入内存；
- 本笔记本不执行任何网络下载。


In [ ]:
from __future__ import annotations

import csv
import json
import platform
import sys
from collections import Counter
from pathlib import Path

SEED = 7
PREVIEW_LINE_LIMIT = 20
PREVIEW_CHAR_LIMIT = 160

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"D:\Courses\nlp-project"),
]
PROJECT_ROOT = next(
    path.resolve()
    for path in candidates
    if (path / "data" / "manifests" / "datasets.csv").exists()
)
MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "datasets.csv"

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "project_root": str(PROJECT_ROOT),
    "manifest": str(MANIFEST_PATH),
})


## 实验计划

1. 读取候选数据集登记表；
2. 检查必填字段和重复 ID；
3. 汇总任务、访问方式和当前状态；
4. 检查已登记的本地文件是否存在；
5. 记录需要在下载前解决的问题。

当前假设：最早应获取小型评测集和可人工审查的小语料，而不是直接下载 CUTE 或模型权重。


In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fieldnames = reader.fieldnames or []

summary = {
    "dataset_count": len(rows),
    "task_counts": dict(Counter(row["task"] for row in rows)),
    "access_counts": dict(Counter(row["access"] for row in rows)),
    "status_counts": dict(Counter(row["status"] for row in rows)),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
required_fields = {
    "dataset_id", "name", "version", "task", "language_pair",
    "domain", "source_style", "data_origin", "source_url",
    "license", "access", "allowed_use", "split_policy",
    "status", "notes",
}
missing_columns = sorted(required_fields - set(fieldnames))
id_counts = Counter(row["dataset_id"] for row in rows)
duplicate_ids = sorted(key for key, count in id_counts.items() if count > 1)
blank_required = []
for row_number, row in enumerate(rows, start=2):
    for field in sorted(required_fields):
        if not row.get(field, "").strip():
            blank_required.append({
                "row": row_number,
                "dataset_id": row.get("dataset_id", ""),
                "field": field,
            })

validation = {
    "missing_columns": missing_columns,
    "duplicate_ids": duplicate_ids,
    "blank_required_count": len(blank_required),
    "blank_required_preview": blank_required[:10],
}
print(json.dumps(validation, ensure_ascii=False, indent=2))


In [ ]:
local_status = []
for row in rows:
    relative_path = row.get("local_path", "").strip()
    local_file = PROJECT_ROOT / relative_path if relative_path else None
    local_status.append({
        "dataset_id": row["dataset_id"],
        "status": row["status"],
        "local_path_registered": bool(relative_path),
        "local_path_exists": bool(local_file and local_file.exists()),
    })

print(json.dumps(local_status, ensure_ascii=False, indent=2))


## 运行后填写：结果与决策

- 登记表是否通过结构检查：
- 需要优先核验的许可证：
- 第一批准备下载的数据：
- 发现的风险或疑问：
- 决策：继续、修正登记表或暂停：


## 下一步

阶段 1 将只获取 FLORES、MITRA 小样本、OpenPecha、Modern Tibetan Corpus 和 PublicCorpus 样例。每个来源先人工检查至少 20 条，再决定是否扩大下载。

在开始阶段 1 前，应先释放更多内存；建议至少保留 4 GB 可用内存，最好 8 GB。
